## Background
This Jupiter Notebook aims to set the baseline for manipulation classification task. \
At first data is preprocessed, including lowercasing, lemmatization, removal of emojis, punctuation, and URLs. Then texts are vectorized using TF-IDF. Then 5 models will be tested: Logistic Regression, Random Forest, XGBoost, and LightGBM. For the last two hyperparameter finetuning will be also applied.

## Imports

In [ ]:
import os
import re
import warnings
import emoji
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt

In [2]:
warnings.filterwarnings("ignore")

## Constants

In [3]:
TRAIN_PATH = "../data/"
TRAIN_NAME = "train.parquet"

## Read data

In [4]:
df = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3822 entries, 0 to 3821
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3822 non-null   object
 1   content        3822 non-null   object
 2   lang           3822 non-null   object
 3   manipulative   3822 non-null   bool  
 4   techniques     2589 non-null   object
 5   trigger_words  2589 non-null   object
dtypes: bool(1), object(5)
memory usage: 153.2+ KB


In [6]:
# If there are no manipulations in the text, we will set techniques and trigger_words to empty lists
df['techniques'] = df['techniques'].apply(lambda x: [] if x is None else x)
df['trigger_words'] = df['trigger_words'].apply(lambda x: [] if x is None else x)

In [7]:
# Number of occurrences of each technique
technique_counts = Counter([tech for sublist in df['techniques'] for tech in sublist])
technique_counts

Counter({'loaded_language': 1973,
         'cherry_picking': 512,
         'glittering_generalities': 483,
         'cliche': 463,
         'euphoria': 462,
         'fud': 385,
         'appeal_to_fear': 300,
         'whataboutism': 158,
         'bandwagon': 157,
         'straw_man': 138})

## Data preprocessing

### One-hot encoding of manipulation techniques

In [8]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['techniques'])
Y


array([[0, 0, 0, ..., 1, 0, 0],
       [0, 0, 1, ..., 1, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 1]])

In [9]:
mlb.classes_

array(['appeal_to_fear', 'bandwagon', 'cherry_picking', 'cliche',
       'euphoria', 'fud', 'glittering_generalities', 'loaded_language',
       'straw_man', 'whataboutism'], dtype=object)

### Spacy models setup

In [10]:
spacy.prefer_gpu()

True

In [11]:
# Download spacy models
!python -m spacy download uk_core_news_lg -q
!python -m spacy download ru_core_news_lg -q


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_lg')

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_lg')


In [12]:
# Load spaCy NER models
nlp_uk = spacy.load("uk_core_news_lg")
nlp_ru = spacy.load("ru_core_news_lg")

In [13]:
# Define stopwords sets (from spaCy)
stopwords_uk = nlp_uk.Defaults.stop_words
stopwords_ru = nlp_ru.Defaults.stop_words

In [14]:
# Define a dictionary for acronym expansion (Ukrainian & Russian)
acronym_dict = {
    "сша": "сполучені штати америки",
    "лол": "дуже смішно",
    "омг": "о боже",
    "бзв": "до вашого відома",
    "незнаю": "не знаю",  # Common typo fix

    "ссср": "союз советских социалистических республик",
    "нло": "неопознанный летающий объект",
    "мчс": "министерство чрезвычайных ситуаций",
}

### Preprocessing

In [15]:
def preprocess_text(text, lang):
    if lang == "ru":
        nlp = nlp_ru
        stopwords_set = stopwords_ru
    else:
        nlp = nlp_uk
        stopwords_set = stopwords_uk

    # Convert to lowercase
    text = text.lower()

    # Expand acronyms
    for acronym, expanded in acronym_dict.items():
        text = re.sub(r'\b' + re.escape(acronym) + r'\b', expanded, text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Remove emojis
    text = emoji.replace_emoji(text, replace='')

    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = text.replace('\n', ' ')    # Replace newline characters with a space
    text = text.strip()               # Remove leading/trailing spaces

    # Process text using spaCy (tokenization & lemmatization)
    doc = nlp(text)
    processed_words = [
        token.lemma_ for token in doc if token.text not in stopwords_set and not token.is_punct
    ]

    return ' '.join(processed_words)

In [16]:
df['clean_content'] = df.apply(lambda row: preprocess_text(row['content'], lang=row['lang']), axis=1)

## TF-IDF embeddings

In [17]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_content'])

# Modeling

In [18]:
# Split dataset
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
models = {
    "Logistic Regression": MultiOutputClassifier(LogisticRegression(max_iter=1000)),
    "Random Forest": MultiOutputClassifier(RandomForestClassifier(n_estimators=100)),
    "XGBoost": MultiOutputClassifier(xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
    "LightGBM": MultiOutputClassifier(lgb.LGBMClassifier())
}

## Initial

In [ ]:
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    print(f"\n{name} Classification Report:\n")
    print(classification_report(Y_test, Y_pred, target_names=mlb.classes_))

Training Logistic Regression...

Logistic Regression Classification Report:

                         precision    recall  f1-score   support

         appeal_to_fear       0.00      0.00      0.00        58
              bandwagon       0.00      0.00      0.00        35
         cherry_picking       0.00      0.00      0.00        97
                 cliche       0.00      0.00      0.00        93
               euphoria       1.00      0.03      0.05        77
                    fud       1.00      0.01      0.03        75
glittering_generalities       0.93      0.13      0.23        97
        loaded_language       0.66      0.76      0.71       392
              straw_man       0.00      0.00      0.00        25
           whataboutism       0.00      0.00      0.00        34

              micro avg       0.67      0.32      0.43       983
              macro avg       0.36      0.09      0.10       983
           weighted avg       0.51      0.32      0.31       983
           

## Hyperparameter tuning

Since LightGBM and XGBoost showed much better results than Logistic regression and Random Forest, we will try to modify their hyperparameters and see whether the results can be even better.

In [23]:
# Hyperparameter tuning for XGBoost
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
param_grid_xgb = {
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7]
}

In [24]:
grid_search_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='f1_macro', verbose=2, n_jobs=-1)
grid_search_xgb.fit(X_train.toarray(), Y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

In [ ]:
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)

y_pred_xgb = grid_search_xgb.best_estimator_.predict(X_test)
print("\nXGBoost Classification Report:\n")
print(classification_report(Y_test, y_pred_xgb, target_names=mlb.classes_))

In [ ]:
# Hyperparameter tuning for LightGBM
lgb_model = lgb.LGBMClassifier(device='gpu')
param_grid_lgb = {
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300, 500],
    'num_leaves': [31, 50, 70]
}

In [ ]:
grid_search_lgb = GridSearchCV(lgb_model, param_grid_lgb, cv=5, scoring='f1_macro', verbose=2, n_jobs=-1)
grid_search_lgb.fit(X_train.toarray(), Y_train)

In [ ]:
print("Best parameters for LightGBM:", grid_search_lgb.best_params_)

y_pred_lgb = grid_search_lgb.best_estimator_.predict(X_test)
print("\nLightGBM Classification Report:\n")
print(classification_report(Y_test, y_pred_lgb, target_names=mlb.classes_))


Fitting 3 folds for each of 27 candidates, totalling 81 fits


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.